# 00 — Industrial Defect Detection: Problem Definition

**Author:** Sherif Abd El-Rady — Computer Vision Engineer

---

## Business problem

Industrial surface inspection is one of the most commercially valuable computer vision applications. A single undetected defect reaching a customer costs orders of magnitude more than catching it during manufacturing. Traditional rule-based vision systems require manual tuning per product and fail on novel defect types.

The core ML challenge: **anomaly detection under severe class imbalance**.

| Property | Value |
|---|---|
| Training data | Normal images only (defects too rare to collect at scale) |
| Test data | Normal + multiple defect types, often unseen at training time |
| Required output | Image-level pass/fail + pixel-level defect localisation |
| Ground truth | Binary mask per anomaly image (MVTec AD dataset) |

## Approach

We implement and compare two methods:

| Method | Description | Trainable params |
|---|---|---|
| **Classical baseline** | Pixel-mean template + SSIM deviation | 0 |
| **PatchCore** | Deep CNN features + coreset k-NN memory bank | 0 (backbone frozen) |

## Dataset: MVTec AD

- **15 categories** of industrial surfaces (textures and objects)
- **5,354 normal** training images
- **1,725 test images** (normal + 70 defect types)
- Pixel-level ground-truth masks for all defect images
- License: Creative Commons Attribution-NonCommercial-ShareAlike 4.0
  → **Research use only. Not for commercial redistribution.**
- Paper: Bergmann et al. (2019). MVTec AD — A Comprehensive Real-World Dataset
  for Unsupervised Anomaly Detection. CVPR 2019.
  https://www.mvtec.com/company/research/datasets/mvtec-ad

## Pipeline

```
Normal training images
         │
         ▼
  WideResNet-50-2 backbone (frozen, ImageNet pretrained)
  Feature extraction: layer2 + layer3  →  concatenated patch embeddings
         │
         ▼
  Greedy coreset subsampling  →  Memory bank (1% of all patches)
         │
Test images ─────────────────────────────────────────────────────┐
         │                                                        │
         ▼                                                        │
  Extract patches  →  k-NN distance to memory bank               │
         │                                                        │
         ▼                                                        │
  Anomaly map (spatial)  +  Image score (max)  ─────────────────►│
                                                        Evaluation│
                                              ROC-AUC / AP / IoU ◄┘
```


In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

import defectdetect

print(f'defectdetect : {defectdetect.__version__}')
print(f'PyTorch      : {torch.__version__}')
print(f'torchvision  : {torchvision.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')
print(f'Device       : {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# ── Dataset acquisition instructions ─────────────────────────────────────────
# MVTec AD must be downloaded manually (license: CC BY-NC-SA 4.0)
# 
# 1. Visit: https://www.mvtec.com/company/research/datasets/mvtec-ad
# 2. Accept the license and download mvtec_anomaly_detection.tar.xz (~4.9 GB)
# 3. Extract to data/mvtec/
#
# Expected structure:
#   data/mvtec/bottle/train/good/*.png
#   data/mvtec/bottle/test/good/*.png
#   data/mvtec/bottle/test/broken_large/*.png
#   data/mvtec/bottle/ground_truth/broken_large/*.png
#
# For this notebook, we use 'bottle' category as the running example.
# The pipeline works identically on any other category.

ROOT     = Path('..').resolve()
MVTEC    = ROOT / 'data' / 'mvtec'
CATEGORY = 'bottle'

train_dir = MVTEC / CATEGORY / 'train' / 'good'
test_dir  = MVTEC / CATEGORY / 'test'
gt_dir    = MVTEC / CATEGORY / 'ground_truth'

if train_dir.exists():
    n_train = len(list(train_dir.glob('*.png')))
    n_test  = sum(len(list(d.glob('*.png'))) for d in test_dir.iterdir() if d.is_dir())
    print(f'✓ MVTec {CATEGORY}: {n_train} train  |  {n_test} test')
else:
    print(f'⚠  MVTec not found at {MVTEC}')
    print('   Download from https://www.mvtec.com/company/research/datasets/mvtec-ad')
    print('   Notebooks 02–08 use synthetic stand-ins if the real dataset is unavailable.')

In [ ]:
# ── Smoke test: WideResNet-50-2 feature extraction ────────────────────────────
from defectdetect import FeatureExtractor

extractor = FeatureExtractor(layers=['layer2', 'layer3'], input_size=224, device='cpu')

# One random 'image' — validates backbone, hooks, and preprocessing
dummy_bgr = (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
batch     = extractor.preprocess([dummy_bgr])
patches, spatial = extractor.extract_patches(batch)

print(f'Input batch shape   : {tuple(batch.shape)}')
print(f'Patch embeddings    : {patches.shape}  (n_patches × feature_dim)')
print(f'Feature map spatial : {spatial}  (H_f × W_f)')
print(f'\n✓ FeatureExtractor smoke test passed')